# Exercice 19 - Pipeline Complet Bronze/Silver/Gold

## Objectifs
- Construire un pipeline ETL complet
- Implementer l'architecture Medallion (Bronze/Silver/Gold)
- Orchestrer les differentes etapes
- Produire des donnees prets pour l'analyse

---

## 1. Architecture du Pipeline

```
+=========================================================================+
|                    PIPELINE MEDALLION COMPLET                           |
+=========================================================================+
|                                                                         |
|  SOURCES                                                                |
|  +-------------+     +-------------+     +-------------+                |
|  | PostgreSQL  |     |    Kafka    |     |   Fichiers  |                |
|  | (Northwind) |     | (Streaming) |     |   (CSV)     |                |
|  +------+------+     +------+------+     +------+------+                |
|         |                   |                   |                       |
|         +-------------------+-------------------+                       |
|                             |                                           |
|                             v                                           |
|  BRONZE (Donnees brutes)                                                |
|  +------------------------------------------------------------------+   |
|  |  s3a://bronze/                                                   |   |
|  |  - customers/    - orders/    - products/    - kafka/            |   |
|  |  Format: Parquet, partition par date d'ingestion                 |   |
|  +------------------------------------------------------------------+   |
|                             |                                           |
|                             v                                           |
|  SILVER (Donnees nettoyees)                                             |
|  +------------------------------------------------------------------+   |
|  |  s3a://silver/                                                   |   |
|  |  - dim_customers/  - dim_products/  - fact_orders/               |   |
|  |  Nettoyage, deduplication, enrichissement                        |   |
|  +------------------------------------------------------------------+   |
|                             |                                           |
|                             v                                           |
|  GOLD (Donnees agregees)                                                |
|  +------------------------------------------------------------------+   |
|  |  s3a://gold/                                                     |   |
|  |  - kpi_ventes/  - analyse_clients/  - rapport_produits/          |   |
|  |  KPIs, aggregations, donnees prets pour dashboards               |   |
|  +------------------------------------------------------------------+   |
|                                                                         |
+=========================================================================+
```

## 2. Configuration

In [142]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, date_format, year, month, dayofmonth,
    when, coalesce, trim, upper, lower, initcap,
    count, sum as spark_sum, avg, max as spark_max, min as spark_min,
    round as spark_round, datediff, current_date,
    row_number, dense_rank, percent_rank
)
from pyspark.sql.window import Window
from datetime import datetime

spark = SparkSession.builder \
    .appName("PipelineComplet") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Configuration
JDBC_URL = "jdbc:postgresql://postgres:5432/app"
JDBC_PROPS = {"user": "postgres", "password": "postgres", "driver": "org.postgresql.Driver"}

BRONZE = "s3a://bronze"
SILVER = "s3a://silver"
GOLD = "s3a://gold"

DATE_INGESTION = datetime.now().strftime("%Y-%m-%d")

print(f"Spark version: {spark.version}")
print(f"Date d'ingestion: {DATE_INGESTION}")

Spark version: 4.1.1
Date d'ingestion: 2026-03-27


## 3. BRONZE - Ingestion des donnees brutes

In [143]:
def ingerer_table_bronze(table_name):
    """
    Ingere une table PostgreSQL vers Bronze.
    Ajoute les metadonnees d'ingestion.
    """
    print(f"Ingestion de {table_name}...")
    
    # Lire depuis PostgreSQL
    df = spark.read.jdbc(
        url=JDBC_URL,
        table=table_name,
        properties=JDBC_PROPS
    )
    
    # Ajouter metadonnees
    df_bronze = df \
        .withColumn("_ingestion_timestamp", current_timestamp()) \
        .withColumn("_source", lit("postgresql")) \
        .withColumn("_source_table", lit(table_name))
    
    # Sauvegarder en Bronze
    output_path = f"{BRONZE}/{table_name}/date={DATE_INGESTION}"
    df_bronze.write.mode("overwrite").parquet(output_path)
    
    nb_lignes = df_bronze.count()
    print(f"  -> {nb_lignes} lignes sauvegardees")
    
    return nb_lignes

In [144]:
# Ingerer les tables principales
tables = ["customers", "orders", "order_details", "products", "categories", "employees"]

print("=" * 50)
print("ETAPE BRONZE - Ingestion des donnees brutes")
print("=" * 50)

stats_bronze = {}
for table in tables:
    stats_bronze[table] = ingerer_table_bronze(table)

print("\nResume Bronze:")
for table, count in stats_bronze.items():
    print(f"  {table}: {count} lignes")

ETAPE BRONZE - Ingestion des donnees brutes
Ingestion de customers...
  -> 91 lignes sauvegardees
Ingestion de orders...
  -> 830 lignes sauvegardees
Ingestion de order_details...
  -> 2155 lignes sauvegardees
Ingestion de products...
  -> 77 lignes sauvegardees
Ingestion de categories...
  -> 8 lignes sauvegardees
Ingestion de employees...
  -> 9 lignes sauvegardees

Resume Bronze:
  customers: 91 lignes
  orders: 830 lignes
  order_details: 2155 lignes
  products: 77 lignes
  categories: 8 lignes
  employees: 9 lignes


## 4. SILVER - Nettoyage et transformation

In [145]:
# Lire les donnees Bronze
df_customers_raw = spark.read.parquet(f"{BRONZE}/customers/date={DATE_INGESTION}")
df_orders_raw = spark.read.parquet(f"{BRONZE}/orders/date={DATE_INGESTION}")
df_order_details_raw = spark.read.parquet(f"{BRONZE}/order_details/date={DATE_INGESTION}")
df_products_raw = spark.read.parquet(f"{BRONZE}/products/date={DATE_INGESTION}")
df_categories_raw = spark.read.parquet(f"{BRONZE}/categories/date={DATE_INGESTION}")
df_employees_raw = spark.read.parquet(f"{BRONZE}/employees/date={DATE_INGESTION}")

print("Donnees Bronze chargees")

Donnees Bronze chargees


In [146]:
print("=" * 50)
print("ETAPE SILVER - Nettoyage et transformation")
print("=" * 50)

# --- DIM_CUSTOMERS ---
print("\nCreation de dim_customers...")

df_dim_customers = df_customers_raw \
    .select(
        col("customer_id"),
        initcap(col("company_name")).alias("company_name"),
        initcap(col("contact_name")).alias("contact_name"),
        col("contact_title"),
        trim(col("address")).alias("address"),
        initcap(col("city")).alias("city"),
        upper(col("region")).alias("region"),
        col("postal_code"),
        initcap(col("country")).alias("country"),
        col("phone"),
        col("fax")
    ) \
    .withColumn("_processed_timestamp", current_timestamp()) \
    .dropDuplicates(["customer_id"])

df_dim_customers.write.mode("overwrite").parquet(f"{SILVER}/dim_customers")
print(f"  -> {df_dim_customers.count()} clients")

ETAPE SILVER - Nettoyage et transformation

Creation de dim_customers...
  -> 91 clients


In [147]:
# --- DIM_PRODUCTS ---
print("\nCreation de dim_products...")

df_dim_products = df_products_raw \
    .join(df_categories_raw, "category_id", "left") \
    .select(
        col("product_id"),
        initcap(df_products_raw["product_name"]).alias("product_name"),
        col("category_id"),
        initcap(col("category_name")).alias("category_name"),
        col("quantity_per_unit"),
        col("unit_price"),
        col("units_in_stock"),
        col("units_on_order"),
        col("reorder_level"),
        col("discontinued"),
        # Indicateurs derives
        when(col("units_in_stock") < col("reorder_level"), lit("Critique"))
            .when(col("units_in_stock") < col("reorder_level") * 2, lit("Bas"))
            .otherwise(lit("Normal")).alias("stock_status"),
        when(col("unit_price") >= 50, lit("Premium"))
            .when(col("unit_price") >= 20, lit("Standard"))
            .otherwise(lit("Budget")).alias("price_segment")
    ) \
    .withColumn("_processed_timestamp", current_timestamp()) \
    .dropDuplicates(["product_id"])

df_dim_products.write.mode("overwrite").parquet(f"{SILVER}/dim_products")
print(f"  -> {df_dim_products.count()} produits")


Creation de dim_products...
  -> 77 produits


In [149]:
# --- DIM_EMPLOYEES ---
print("\nCreation de dim_employees...")

from pyspark.sql.functions import concat_ws

df_dim_employees = df_employees_raw \
    .select(
        col("employee_id"),
        initcap(col("first_name")).alias("first_name"),
        initcap(col("last_name")).alias("last_name"),
        col("title"),
        col("hire_date"),
        initcap(col("city")).alias("city"),
        initcap(col("country")).alias("country")
    ) \
    .withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name"))) \
    .withColumn("_processed_timestamp", current_timestamp())

df_dim_employees.write.mode("overwrite").parquet(f"{SILVER}/dim_employees")
print(f"  -> {df_dim_employees.count()} employes")



Creation de dim_employees...
  -> 9 employes


In [150]:
# --- FACT_ORDERS ---
print("\nCreation de fact_orders...")

df_fact_orders = df_orders_raw \
    .join(df_order_details_raw, "order_id", "inner") \
    .select(
        col("order_id"),
        col("customer_id"),
        col("employee_id"),
        col("order_date"),
        col("required_date"),
        col("shipped_date"),
        col("product_id"),
        col("unit_price"),
        col("quantity"),
        col("discount"),
        # Montants calcules
        (col("unit_price") * col("quantity")).alias("montant_brut"),
        (col("unit_price") * col("quantity") * (1 - col("discount"))).alias("montant_net"),
        (col("unit_price") * col("quantity") * col("discount")).alias("remise"),
        # Dimensions temporelles
        year(col("order_date")).alias("annee"),
        month(col("order_date")).alias("mois"),
        dayofmonth(col("order_date")).alias("jour")
    ) \
    .withColumn("_processed_timestamp", current_timestamp())

df_fact_orders.write.mode("overwrite").parquet(f"{SILVER}/fact_orders")
print(f"  -> {df_fact_orders.count()} lignes de commande")


Creation de fact_orders...
  -> 2155 lignes de commande


## 5. GOLD - Agregations et KPIs

In [151]:
# Charger les donnees Silver
df_customers = spark.read.parquet(f"{SILVER}/dim_customers")
df_products = spark.read.parquet(f"{SILVER}/dim_products")
df_employees = spark.read.parquet(f"{SILVER}/dim_employees")
df_orders = spark.read.parquet(f"{SILVER}/fact_orders")

print("Donnees Silver chargees")
print("=" * 50)
print("ETAPE GOLD - Agregations et KPIs")
print("=" * 50)

Donnees Silver chargees
ETAPE GOLD - Agregations et KPIs


In [155]:
# --- KPI VENTES MENSUELLES ---
print("\nCreation de kpi_ventes_mensuelles...")

import pyspark.sql.functions as F

# Then use F.count(), F.avg(), F.sum(), etc.
df_kpi_ventes = df_orders \
    .groupBy("annee", "mois") \
    .agg(
        F.count("order_id").alias("nb_commandes"),
        F.countDistinct("customer_id").alias("nb_clients"),
        F.sum("montant_net").alias("ca_total"),
        F.avg("montant_net").alias("panier_moyen"),
        F.sum("quantity").alias("quantite_totale"),
        F.sum("remise").alias("remises_totales")
    ) \
    .withColumn("ca_total", F.round(F.col("ca_total"), 2)) \
    .withColumn("panier_moyen", F.round(F.col("panier_moyen"), 2)) \
    .withColumn("remises_totales", F.round(F.col("remises_totales"), 2)) \
    .orderBy("annee", "mois") \
    .withColumn("_generated_timestamp", F.current_timestamp())

df_kpi_ventes.write.mode("overwrite").parquet(f"{GOLD}/kpi_ventes_mensuelles")
print(f"  -> {df_kpi_ventes.count()} mois")
df_kpi_ventes.show()



Creation de kpi_ventes_mensuelles...
  -> 23 mois
+-----+----+------------+----------+--------+------------+---------------+---------------+--------------------+
|annee|mois|nb_commandes|nb_clients|ca_total|panier_moyen|quantite_totale|remises_totales|_generated_timestamp|
+-----+----+------------+----------+--------+------------+---------------+---------------+--------------------+
| 1996|   7|          59|        20| 27861.9|      472.24|           1462|        2330.21|2026-03-27 14:18:...|
| 1996|   8|          69|        18|25485.28|      369.35|           1322|        1124.13|2026-03-27 14:18:...|
| 1996|   9|          57|        19| 26381.4|      462.83|           1124|         1254.6|2026-03-27 14:18:...|
| 1996|  10|          73|        20|37515.72|      513.91|           1738|        3687.88|2026-03-27 14:18:...|
| 1996|  11|          66|        21|45600.05|      690.91|           1735|        4103.96|2026-03-27 14:18:...|
| 1996|  12|          81|        25|45239.63|      55

In [157]:
# --- ANALYSE CLIENTS ---
print("\nCreation de analyse_clients...")

from pyspark.sql.functions import count  # ← add this line

df_analyse_clients = df_orders \
    .groupBy("customer_id") \
    .agg(
        count("order_id").alias("nb_commandes"),
        spark_sum("montant_net").alias("ca_total"),
        avg("montant_net").alias("panier_moyen"),
        spark_min("order_date").alias("premiere_commande"),
        spark_max("order_date").alias("derniere_commande")
    ) \
    .join(df_customers.select("customer_id", "company_name", "country", "city"), "customer_id") \
    .withColumn("ca_total", spark_round(col("ca_total"), 2)) \
    .withColumn("panier_moyen", spark_round(col("panier_moyen"), 2))

# Segmentation RFM simplifiee
window_ca = Window.orderBy(col("ca_total").desc())

df_analyse_clients = df_analyse_clients \
    .withColumn("rang_ca", dense_rank().over(window_ca)) \
    .withColumn("segment",
        when(col("rang_ca") <= 10, lit("VIP"))
        .when(col("rang_ca") <= 30, lit("Premium"))
        .when(col("rang_ca") <= 60, lit("Standard"))
        .otherwise(lit("Occasionnel"))
    ) \
    .withColumn("_generated_timestamp", current_timestamp())

df_analyse_clients.write.mode("overwrite").parquet(f"{GOLD}/analyse_clients")
print(f"  -> {df_analyse_clients.count()} clients analyses")
df_analyse_clients.select(
    "company_name", "country", "nb_commandes", "ca_total", "segment"
).orderBy(col("ca_total").desc()).show(10)


Creation de analyse_clients...
  -> 89 clients analyses
+--------------------+-------+------------+---------+-------+
|        company_name|country|nb_commandes| ca_total|segment|
+--------------------+-------+------------+---------+-------+
|          Quick-stop|Germany|          86|110277.31|    VIP|
|        Ernst Handel|Austria|         102|104874.98|    VIP|
|  Save-a-lot Markets|    Usa|         116|104361.95|    VIP|
|Rattlesnake Canyo...|    Usa|          71|  51097.8|    VIP|
|Hungry Owl All-ni...|Ireland|          55| 49979.91|    VIP|
|       Hanari Carnes| Brazil|          32| 32841.37|    VIP|
|     Königlich Essen|Germany|          39| 30908.38|    VIP|
|      Folk Och Fä Hb| Sweden|          45| 29567.56|    VIP|
|      Mère Paillarde| Canada|          32| 28872.19|    VIP|
|White Clover Markets|    Usa|          40|  27363.6|    VIP|
+--------------------+-------+------------+---------+-------+
only showing top 10 rows


In [158]:
# --- RAPPORT PRODUITS ---
print("\nCreation de rapport_produits...")

df_rapport_produits = df_orders \
    .groupBy("product_id") \
    .agg(
        count("order_id").alias("nb_ventes"),
        spark_sum("quantity").alias("quantite_vendue"),
        spark_sum("montant_net").alias("ca_produit"),
        avg("unit_price").alias("prix_moyen")
    ) \
    .join(df_products.select(
        "product_id", "product_name", "category_name", 
        "stock_status", "price_segment", "units_in_stock"
    ), "product_id") \
    .withColumn("ca_produit", spark_round(col("ca_produit"), 2)) \
    .withColumn("prix_moyen", spark_round(col("prix_moyen"), 2))

# Ranking par categorie
window_cat = Window.partitionBy("category_name").orderBy(col("ca_produit").desc())

df_rapport_produits = df_rapport_produits \
    .withColumn("rang_categorie", row_number().over(window_cat)) \
    .withColumn("_generated_timestamp", current_timestamp())

df_rapport_produits.write.mode("overwrite").parquet(f"{GOLD}/rapport_produits")
print(f"  -> {df_rapport_produits.count()} produits analyses")
df_rapport_produits.select(
    "product_name", "category_name", "quantite_vendue", "ca_produit", "rang_categorie"
).orderBy(col("ca_produit").desc()).show(10)


Creation de rapport_produits...
  -> 77 produits analyses
+--------------------+--------------+---------------+----------+--------------+
|        product_name| category_name|quantite_vendue|ca_produit|rang_categorie|
+--------------------+--------------+---------------+----------+--------------+
|       Côte De Blaye|     Beverages|            623| 141396.74|             1|
|Thüringer Rostbra...|  Meat/poultry|            746|  80368.67|             1|
|Raclette Courdavault|Dairy Products|           1496|   71155.7|             1|
|      Tarte Au Sucre|   Confections|           1083|  47234.97|             1|
|   Camembert Pierrot|Dairy Products|           1577|  46825.48|             2|
|Gnocchi Di Nonna ...|Grains/cereals|           1263|  42593.06|             1|
|Manjimup Dried Ap...|       Produce|            886|  41819.65|             1|
|        Alice Mutton|  Meat/poultry|            978|  32698.38|             2|
|    Carnarvon Tigers|       Seafood|            539|  29171.

In [159]:
# --- PERFORMANCE EMPLOYES ---
print("\nCreation de performance_employes...")

df_perf_employes = df_orders \
    .groupBy("employee_id") \
    .agg(
        countDistinct("order_id").alias("nb_commandes"),
        countDistinct("customer_id").alias("nb_clients"),
        spark_sum("montant_net").alias("ca_genere"),
        avg("montant_net").alias("ca_moyen_ligne")
    ) \
    .join(df_employees.select("employee_id", "full_name", "title", "city"), "employee_id") \
    .withColumn("ca_genere", spark_round(col("ca_genere"), 2)) \
    .withColumn("ca_moyen_ligne", spark_round(col("ca_moyen_ligne"), 2)) \
    .withColumn("_generated_timestamp", current_timestamp())

df_perf_employes.write.mode("overwrite").parquet(f"{GOLD}/performance_employes")
print(f"  -> {df_perf_employes.count()} employes")
df_perf_employes.orderBy(col("ca_genere").desc()).show()


Creation de performance_employes...
  -> 9 employes
+-----------+------------+----------+---------+--------------+----------------+--------------------+--------+--------------------+
|employee_id|nb_commandes|nb_clients|ca_genere|ca_moyen_ligne|       full_name|               title|    city|_generated_timestamp|
+-----------+------------+----------+---------+--------------+----------------+--------------------+--------+--------------------+
|          4|         156|        75|232890.85|         554.5|Margaret Peacock|Sales Representative| Redmond|2026-03-27 14:20:...|
|          3|         127|        63|202812.84|        631.82| Janet Leverling|Sales Representative|Kirkland|2026-03-27 14:20:...|
|          1|         123|        65| 192107.6|        556.83|   Nancy Davolio|Sales Representative| Seattle|2026-03-27 14:20:...|
|          2|          96|        59|166537.76|        691.03|   Andrew Fuller|Vice President, S...|  Tacoma|2026-03-27 14:20:...|
|          8|         104|    

## 6. Verification du pipeline

In [160]:
print("=" * 60)
print("VERIFICATION DU PIPELINE")
print("=" * 60)

# Verifier Bronze
print("\nBRONZE:")
for table in tables:
    try:
        row_count = spark.read.parquet(f"{BRONZE}/{table}/date={DATE_INGESTION}").count()
        print(f"  {table}: {row_count} lignes")
    except:
        print(f"  {table}: ERREUR")

# Verifier Silver
print("\nSILVER:")
for dim in ["dim_customers", "dim_products", "dim_employees", "fact_orders"]:
    try:
        count = spark.read.parquet(f"{SILVER}/{dim}").count()
        print(f"  {dim}: {count} lignes")
    except:
        print(f"  {dim}: ERREUR")

# Verifier Gold
print("\nGOLD:")
for kpi in ["kpi_ventes_mensuelles", "analyse_clients", "rapport_produits", "performance_employes"]:
    try:
        count = spark.read.parquet(f"{GOLD}/{kpi}").count()
        print(f"  {kpi}: {count} lignes")
    except:
        print(f"  {kpi}: ERREUR")

VERIFICATION DU PIPELINE

BRONZE:
  customers: 91 lignes
  orders: 830 lignes
  order_details: 2155 lignes
  products: 77 lignes
  categories: 8 lignes
  employees: 9 lignes

SILVER:
  dim_customers: 91 lignes
  dim_products: 77 lignes
  dim_employees: 9 lignes
  fact_orders: 2155 lignes

GOLD:
  kpi_ventes_mensuelles: 23 lignes
  analyse_clients: 89 lignes
  rapport_produits: 77 lignes
  performance_employes: 9 lignes


In [161]:
# Resume executif
print("\n" + "=" * 60)
print("RESUME EXECUTIF")
print("=" * 60)

# KPIs globaux
df_kpi = spark.read.parquet(f"{GOLD}/kpi_ventes_mensuelles")
total_ca = df_kpi.agg(spark_sum("ca_total")).collect()[0][0]
total_commandes = df_kpi.agg(spark_sum("nb_commandes")).collect()[0][0]

df_clients = spark.read.parquet(f"{GOLD}/analyse_clients")
nb_clients = df_clients.count()
nb_vip = df_clients.filter(col("segment") == "VIP").count()

print(f"\nChiffre d'affaires total: {total_ca:,.2f} EUR")
print(f"Nombre de commandes: {total_commandes:,}")
print(f"Nombre de clients: {nb_clients}")
print(f"Clients VIP: {nb_vip}")
print(f"\nPipeline execute le: {DATE_INGESTION}")


RESUME EXECUTIF

Chiffre d'affaires total: 1,265,793.05 EUR
Nombre de commandes: 2,155
Nombre de clients: 89
Clients VIP: 10

Pipeline execute le: 2026-03-27


---

## Exercice

**Objectif** : Etendre le pipeline

**Consigne** :
1. Ajoutez une table Gold "analyse_geographique" avec le CA par pays
2. Calculez la croissance mensuelle du CA
3. Identifiez le top 3 des pays par CA

A vous de jouer :

In [163]:
# --- ANALYSE GEOGRAPHIQUE ---
print("\nCreation de analyse_geographique...")

from pyspark.sql.functions import count

# Join avec df_customers pour recuperer country et city
df_orders_geo = df_orders \
    .join(df_customers.select("customer_id", "country", "city"), "customer_id")

df_analyse_geo = df_orders_geo \
    .groupBy("country", "city") \
    .agg(
        count("order_id").alias("nb_commandes"),
        countDistinct("customer_id").alias("nb_clients"),
        spark_sum("montant_net").alias("ca_total"),
        avg("montant_net").alias("panier_moyen"),
        spark_sum("quantity").alias("quantite_totale")
    ) \
    .withColumn("ca_total", spark_round(col("ca_total"), 2)) \
    .withColumn("panier_moyen", spark_round(col("panier_moyen"), 2)) \
    .withColumn("_generated_timestamp", current_timestamp())

# Agregation par pays
df_analyse_pays = df_orders_geo \
    .groupBy("country") \
    .agg(
        count("order_id").alias("nb_commandes"),
        countDistinct("customer_id").alias("nb_clients"),
        spark_sum("montant_net").alias("ca_total"),
        spark_sum("quantity").alias("quantite_totale")
    ) \
    .withColumn("ca_total", spark_round(col("ca_total"), 2)) \
    .withColumn("pct_ca",
        spark_round(
            col("ca_total") / spark_sum("ca_total").over(
                Window.orderBy(lit(1)).rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
            ) * 100, 2
        )
    ) \
    .orderBy(col("ca_total").desc()) \
    .withColumn("_generated_timestamp", current_timestamp())

df_analyse_geo.write.mode("overwrite").parquet(f"{GOLD}/analyse_geographique")
df_analyse_pays.write.mode("overwrite").parquet(f"{GOLD}/analyse_pays")

print(f"  -> {df_analyse_geo.count()} villes analysees")
print(f"  -> {df_analyse_pays.count()} pays analyses")

print("\nTop 10 pays par CA:")
df_analyse_pays.select("country", "nb_commandes", "nb_clients", "ca_total", "pct_ca").show(10)

print("\nTop 10 villes par CA:")
df_analyse_geo.select("country", "city", "nb_commandes", "ca_total") \
    .orderBy(col("ca_total").desc()) \
    .show(10)


Creation de analyse_geographique...
  -> 69 villes analysees
  -> 21 pays analyses

Top 10 pays par CA:
+---------+------------+----------+---------+------+
|  country|nb_commandes|nb_clients| ca_total|pct_ca|
+---------+------------+----------+---------+------+
|      Usa|         352|        13|245584.61|  19.4|
|  Germany|         328|        11|230284.63| 18.19|
|  Austria|         125|         2|128003.84| 10.11|
|   Brazil|         203|         9|106925.78|  8.45|
|   France|         184|        10| 81358.32|  6.43|
|       Uk|         135|         7| 58971.31|  4.66|
|Venezuela|         118|         4| 56810.63|  4.49|
|   Sweden|          97|         2| 54495.14|  4.31|
|   Canada|          75|         3| 50196.29|  3.97|
|  Ireland|          55|         1| 49979.91|  3.95|
+---------+------------+----------+---------+------+
only showing top 10 rows

Top 10 villes par CA:
+-------+--------------+------------+---------+
|country|          city|nb_commandes| ca_total|
+-------+

---

## Resume

Dans ce notebook, vous avez appris :
- Comment construire un **pipeline ETL complet**
- L'architecture **Medallion** (Bronze/Silver/Gold)
- L'**ingestion** depuis PostgreSQL vers Bronze
- Le **nettoyage et transformation** vers Silver
- La creation de **KPIs et agregations** dans Gold
- La **verification** du pipeline

Ce pipeline est la base de toute architecture Data Lake moderne.